# Structured Pruning of DeblurGAN-v2: Experiment Notebook

This notebook runs the full experiment for my MSc thesis. It loads the pre-trained DeblurGAN-v2 model, measures its quality on the GoPro test set, prunes away the weakest channels at 10%, 30% and 50%, fine tunes each pruned model for 50 epochs, and compares everything at the end.

Before the first run:
1. Runtime > Change runtime type > GPU
2. Put fpn_inception.h5 in your Google Drive root
3. Download GOPRO_Large.zip from https://seungjunnah.github.io/Datasets/gopro and upload it to Drive at MyDrive/datasets/GOPRO_Large.zip (keep it zipped)

The whole experiment takes many hours. Everything gets checkpointed to Drive, so if Colab disconnects just Run All again and it continues from where it stopped. The 30% ratio runs first so any training problem shows up early.


## 1. Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os
if not os.path.exists('DeblurGANv2'):
    !git clone https://github.com/VITA-Group/DeblurGANv2.git
%cd DeblurGANv2

!pip install -q pretrainedmodels==0.7.4 albumentations==1.3.0 gdown fire glog tqdm scikit-image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

All paths and hyperparameters in one place.

In [ ]:
import os, shutil

# Paths
WEIGHTS_PATH = '/content/drive/MyDrive/fpn_inception.h5'
GOPRO_ZIP = '/content/drive/MyDrive/datasets/GOPRO_Large.zip'  # full official set, uploaded once
GOPRO_ROOT = '/content/GOPRO_Large'                            # extracted to local disk each session
# v1 run used wrong loss weights and failed, v2 uses the weights from the paper
V1_CHECKPOINT_BASE = '/content/drive/MyDrive/checkpoints/deblurgan_pruning_full'
CHECKPOINT_BASE = '/content/drive/MyDrive/checkpoints/deblurgan_pruning_full_v2'
FIGURES_DIR = '/content/drive/MyDrive/figures'

# three pruning ratios, 30% goes first to check the loss fix early
PRUNING_RATIOS = [0.30, 0.10, 0.50]

# fine tuning settings
NUM_EPOCHS = 50            # long enough to separate pruning damage from under-training
BATCH_SIZE = 2
LEARNING_RATE_G = 5e-5
LEARNING_RATE_D = 5e-5
CROP_SIZE = 256
# loss weights from the original paper
PIXEL_WEIGHT = 0.5
PERCEPTUAL_WEIGHT = 0.006
ADVERSARIAL_WEIGHT = 0.01

# validate every few epochs and keep the best checkpoint
VAL_EVERY = 5      # validate every N epochs
VAL_IMAGES = 100   # fixed subset of the test set used for periodic validation

os.makedirs(CHECKPOINT_BASE, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# reuse results from v1 that are still valid to save time
if os.path.exists(V1_CHECKPOINT_BASE):
    reusable = [(os.path.join(V1_CHECKPOINT_BASE, 'baseline_results.json'),
                 os.path.join(CHECKPOINT_BASE, 'baseline_results.json'))]
    for r in PRUNING_RATIOS:
        tag = f"ratio_{int(r*100)}pct"
        reusable.append((os.path.join(V1_CHECKPOINT_BASE, tag, 'pre_ft_results.json'),
                         os.path.join(CHECKPOINT_BASE, tag, 'pre_ft_results.json')))
    for src, dst in reusable:
        if os.path.exists(src) and not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            print(f"Reused from v1: {os.path.relpath(dst, CHECKPOINT_BASE)}")

assert os.path.exists(WEIGHTS_PATH), f"Weights not found at {WEIGHTS_PATH}"
print(f"Pruning ratios: {[f'{r*100:.0f}%' for r in PRUNING_RATIOS]}")
print(f"Epochs per ratio: {NUM_EPOCHS} (validation every {VAL_EVERY})")
print("Config OK.")

In [ ]:
# get the GoPro dataset ready, extract to local disk because Drive is too slow for training
import shutil, zipfile

if os.path.exists(os.path.join(GOPRO_ROOT, 'train')):
    print("GoPro dataset already extracted on local disk.")
else:
    assert os.path.exists(GOPRO_ZIP), (
        f"{GOPRO_ZIP} not found. Download GOPRO_Large.zip from "
        "https://seungjunnah.github.io/Datasets/gopro and upload it to that Drive path."
    )
    local_zip = '/content/GOPRO_Large.zip'
    print("Copying zip from Drive to local disk (a few minutes)...")
    shutil.copy2(GOPRO_ZIP, local_zip)
    print("Extracting...")
    extract_dir = '/content/gopro_extract'
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(extract_dir)
    os.remove(local_zip)
    # handle both zip layouts
    inner = os.path.join(extract_dir, 'GOPRO_Large')
    shutil.move(inner if os.path.exists(inner) else extract_dir, GOPRO_ROOT)
    print("Extraction complete.")

for split in ['train', 'test']:
    split_dir = os.path.join(GOPRO_ROOT, split)
    scenes = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
    print(f"  {split}: {len(scenes)} scenes")

## 3. GoPro dataset

In [ ]:
import os, glob, random, numpy as np, cv2, torch
from torch.utils.data import Dataset, DataLoader, Subset

class GoProDataset(Dataset):
    """Loads paired blur/sharp images from the official scene-based GoPro structure."""
    def __init__(self, blur_paths, sharp_paths, crop_size=None):
        self.blur_paths = blur_paths
        self.sharp_paths = sharp_paths
        self.crop_size = crop_size

    def __len__(self):
        return len(self.blur_paths)

    def __getitem__(self, idx):
        blur = cv2.cvtColor(cv2.imread(self.blur_paths[idx]), cv2.COLOR_BGR2RGB)
        sharp = cv2.cvtColor(cv2.imread(self.sharp_paths[idx]), cv2.COLOR_BGR2RGB)

        if self.crop_size:
            h, w = blur.shape[:2]
            top = random.randint(0, h - self.crop_size)
            left = random.randint(0, w - self.crop_size)
            blur = blur[top:top+self.crop_size, left:left+self.crop_size]
            sharp = sharp[top:top+self.crop_size, left:left+self.crop_size]
            if random.random() > 0.5:
                blur = blur[:, ::-1].copy()
                sharp = sharp[:, ::-1].copy()

        blur = blur.astype(np.float32) / 127.5 - 1.0
        sharp = sharp.astype(np.float32) / 127.5 - 1.0
        return (torch.from_numpy(blur.transpose(2, 0, 1)),
                torch.from_numpy(sharp.transpose(2, 0, 1)))

# official split used by all the papers
train_blur = sorted(glob.glob(os.path.join(GOPRO_ROOT, 'train', '*', 'blur', '*.png')))
train_sharp = [p.replace('/blur/', '/sharp/') for p in train_blur]
test_blur = sorted(glob.glob(os.path.join(GOPRO_ROOT, 'test', '*', 'blur', '*.png')))
test_sharp = [p.replace('/blur/', '/sharp/') for p in test_blur]

assert len(train_blur) == 2103, f"Expected 2,103 train pairs, found {len(train_blur)}, check the dataset upload"
assert len(test_blur) == 1111, f"Expected 1,111 test pairs, found {len(test_blur)}, check the dataset upload"
assert os.path.exists(train_sharp[0]) and os.path.exists(test_sharp[0]), "Sharp counterparts missing"

train_dataset = GoProDataset(train_blur, train_sharp, crop_size=CROP_SIZE)
test_dataset = GoProDataset(test_blur, test_sharp, crop_size=None)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

# small fixed subset for quick validation
val_indices = np.linspace(0, len(test_dataset) - 1, VAL_IMAGES, dtype=int).tolist()
val_loader = DataLoader(Subset(test_dataset, val_indices), batch_size=1, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"Test:  {len(test_dataset)} images")
print(f"Val subset: {len(val_indices)} images (every {VAL_EVERY} epochs)")

## 4. Load baseline model

In [ ]:
import sys, os, shutil

sys.path.insert(0, '.')

# skip the imagenet download, the checkpoint overwrites these weights anyway
cache_dir = '/root/.cache/torch/hub/checkpoints'
os.makedirs(cache_dir, exist_ok=True)
imagenet_path = os.path.join(cache_dir, 'inceptionresnetv2-520b38e4.pth')
drive_cache = '/content/drive/MyDrive/model_weights/inceptionresnetv2-520b38e4.pth'

need_patch = False
if not os.path.exists(imagenet_path):
    if os.path.exists(drive_cache):
        shutil.copy2(drive_cache, imagenet_path)
        print(f'Copied ImageNet weights from Drive ({os.path.getsize(imagenet_path)/1e6:.0f} MB)')
    else:
        need_patch = True

if need_patch:
    import pretrainedmodels
    # only patch once, running this cell twice would loop otherwise
    if not getattr(pretrainedmodels.inceptionresnetv2, 'skips_download', False):
        _orig_fn = pretrainedmodels.inceptionresnetv2
        def _no_download(num_classes=1001, pretrained='imagenet'):
            return _orig_fn(num_classes=num_classes, pretrained=None)
        _no_download.skips_download = True
        pretrainedmodels.inceptionresnetv2 = _no_download
    print('Skipping ImageNet download (overwritten by checkpoint)')
else:
    print('ImageNet backbone weights: OK')

from models.networks import get_generator

generator = get_generator({
    'g_name': 'fpn_inception',
    'norm_layer': 'instance',
    'learn_residual': True,
    'dropout': True,
    'blocks': 9,
})

ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
generator.load_state_dict(ckpt['model'])
generator = generator.cuda().eval()

total_params = sum(p.numel() for p in generator.parameters())
print(f'Generator: {total_params:,} parameters ({total_params * 4 / 1e6:.1f} MB at FP32)')

## 5. Evaluation pipeline

In [ ]:
import time
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm import tqdm

def evaluate(model, loader, device='cuda', desc='Evaluating', warmup=5):
    model.eval()
    psnr_list, ssim_list, latency_list = [], [], []
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for blur, sharp in tqdm(loader, desc=desc):
            blur = blur.to(device)
            _, _, h, w = blur.shape

            pad_h = (32 - h % 32) % 32
            pad_w = (32 - w % 32) % 32
            blur_pad = F.pad(blur, (0, pad_w, 0, pad_h), mode='reflect') if (pad_h or pad_w) else blur

            torch.cuda.synchronize()
            t0 = time.time()
            output = model(blur_pad)
            torch.cuda.synchronize()
            latency_list.append((time.time() - t0) * 1000)

            output = output[:, :, :h, :w]
            out_np = ((output.squeeze().cpu().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            gt_np = ((sharp.squeeze().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)

            psnr_list.append(peak_signal_noise_ratio(gt_np, out_np))
            ssim_list.append(structural_similarity(gt_np, out_np, channel_axis=2))

    peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.pth') as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / 1e6

    # skip the warmup iterations
    timed = latency_list[warmup:] if len(latency_list) > warmup else latency_list

    results = {
        'psnr': float(np.mean(psnr_list)),
        'ssim': float(np.mean(ssim_list)),
        'latency_ms': float(np.mean(timed)),
        'peak_memory_gb': float(peak_mem_gb),
        'model_size_mb': float(size_mb),
    }

    print(f"\n{'Metric':<20} {'Value':>10}")
    print("-" * 32)
    for k, v in results.items():
        fmt = '.2f' if 'psnr' in k or 'memory' in k else '.4f' if 'ssim' in k else '.1f'
        print(f"{k:<20} {v:>10{fmt}}")

    return results, psnr_list, ssim_list

def validate(model, loader, device='cuda'):
    """Lightweight PSNR/SSIM check on the fixed validation subset (convergence tracking)."""
    was_training = model.training
    model.eval()
    psnrs, ssims = [], []
    with torch.no_grad():
        for blur, sharp in loader:
            blur = blur.to(device)
            _, _, h, w = blur.shape
            pad_h = (32 - h % 32) % 32
            pad_w = (32 - w % 32) % 32
            blur_pad = F.pad(blur, (0, pad_w, 0, pad_h), mode='reflect') if (pad_h or pad_w) else blur
            out = model(blur_pad)[:, :, :h, :w]
            out_np = ((out.squeeze().cpu().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            gt_np = ((sharp.squeeze().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            psnrs.append(peak_signal_noise_ratio(gt_np, out_np))
            ssims.append(structural_similarity(gt_np, out_np, channel_axis=2))
    if was_training:
        model.train()
    return float(np.mean(psnrs)), float(np.mean(ssims))

print("evaluate() and validate() ready.")

## 6. Baseline evaluation

In [ ]:
import json

# cache the baseline eval so it only runs once
baseline_cache = os.path.join(CHECKPOINT_BASE, 'baseline_results.json')
if os.path.exists(baseline_cache):
    with open(baseline_cache) as f:
        baseline_results = json.load(f)
    print("Loaded cached baseline results:")
    for k, v in baseline_results.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    print("Evaluating unpruned baseline on the full GoPro test set (1,111 images)...")
    baseline_results, _, _ = evaluate(generator, test_loader, desc='Baseline')
    with open(baseline_cache, 'w') as f:
        json.dump(baseline_results, f, indent=2)
    print(f"Saved: {baseline_cache}")

## 7. Multi-ratio pruning experiment

The main loop. For each ratio it copies the baseline, prunes it, checks the quality before fine tuning, fine tunes for 50 epochs and then evaluates the best checkpoint.

A few practical things. Training runs for hours, so a checkpoint goes to Drive after every epoch and finished ratios are skipped when the notebook is re-run. Validation runs every 5 epochs on a small subset and the best model is kept separately.

My first attempt at fine tuning used the wrong loss weights and quality got worse instead of better, so this version uses the loss weights from the original paper. Also the pre-trained discriminator was never released, so a new one is trained from scratch.


In [ ]:
import copy
import torch.nn as nn
import torchvision
import json

# loss components

# perceptual loss on vgg19 features like the paper
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = torchvision.models.vgg19(weights='IMAGENET1K_V1').features[:15].eval()
        self.features = vgg
        for p in self.features.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        x = (x + 1) / 2
        y = (y + 1) / 2
        x = (x - self.mean) / self.std
        y = (y - self.mean) / self.std
        return F.mse_loss(self.features(x), self.features(y))

# patch discriminator trained from scratch, pretrained weights were never released
class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=3, ndf=64, n_layers=3):
        super().__init__()
        layers = [nn.Conv2d(in_channels, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True)]
        ch = ndf
        for i in range(1, n_layers):
            prev = ch
            ch = min(ch * 2, 512)
            layers += [nn.Conv2d(prev, ch, 4, 2, 1), nn.InstanceNorm2d(ch), nn.LeakyReLU(0.2, True)]
        prev = ch
        ch = min(ch * 2, 512)
        layers += [
            nn.Conv2d(prev, ch, 4, 1, 1), nn.InstanceNorm2d(ch), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ch, 1, 4, 1, 1)
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def d_loss_fn(real_pred, fake_pred):
    real = F.binary_cross_entropy_with_logits(real_pred - fake_pred.mean(), torch.ones_like(real_pred))
    fake = F.binary_cross_entropy_with_logits(fake_pred - real_pred.mean(), torch.zeros_like(fake_pred))
    return (real + fake) / 2

def g_loss_fn(real_pred, fake_pred):
    real = F.binary_cross_entropy_with_logits(real_pred - fake_pred.mean(), torch.zeros_like(real_pred))
    fake = F.binary_cross_entropy_with_logits(fake_pred - real_pred.mean(), torch.ones_like(fake_pred))
    return (real + fake) / 2

def apply_masks(model, masks):
    with torch.no_grad():
        for name, module in model.named_modules():
            if name in masks:
                module.weight.data *= masks[name]['weight'].to(module.weight.device)
                if module.bias is not None and masks[name]['bias'] is not None:
                    module.bias.data *= masks[name]['bias'].to(module.bias.device)

# main experiment loop

all_results = {}

for ratio in PRUNING_RATIOS:
    tag = f"{int(ratio*100)}pct"
    ckpt_dir = os.path.join(CHECKPOINT_BASE, f'ratio_{tag}')
    os.makedirs(ckpt_dir, exist_ok=True)

    results_path = os.path.join(ckpt_dir, 'results.json')
    final_model_path = os.path.join(ckpt_dir, f'pruned_{tag}_finetuned.pth')
    latest_ckpt = os.path.join(ckpt_dir, 'checkpoint_latest.pth')
    best_ckpt = os.path.join(ckpt_dir, 'checkpoint_best.pth')
    history_path = os.path.join(ckpt_dir, 'history.json')
    pre_ft_path = os.path.join(ckpt_dir, 'pre_ft_results.json')

    # skip ratios that already finished
    if os.path.exists(results_path) and os.path.exists(final_model_path):
        with open(results_path) as f:
            all_results[tag] = json.load(f)
        print(f"{tag}: already complete, loaded saved results, skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"PRUNING RATIO: {ratio*100:.0f}%")
    print(f"{'='*60}")

    # 1. copy the baseline and zero out the weakest channels
    pruned_gen = copy.deepcopy(generator)
    total_ch, zeroed_ch = 0, 0
    for module in pruned_gen.modules():
        if isinstance(module, nn.Conv2d) and module.out_channels > 3:
            w = module.weight.data
            n_ch = w.shape[0]
            scores = w.abs().sum(dim=[1, 2, 3])
            n_zero = int(n_ch * ratio)
            if n_zero == 0:
                continue
            _, idx = torch.topk(scores, n_zero, largest=False)
            w[idx] = 0
            if module.bias is not None:
                module.bias.data[idx] = 0
            total_ch += n_ch
            zeroed_ch += n_zero

    print(f"Channels zeroed: {zeroed_ch:,} / {total_ch:,} ({zeroed_ch/total_ch*100:.1f}%)")

    # masks keep the pruned channels at zero
    pruning_masks = {}
    for name, module in pruned_gen.named_modules():
        if isinstance(module, nn.Conv2d) and module.out_channels > 3:
            w = module.weight.data
            active = (w.view(w.shape[0], -1).abs().sum(dim=1) > 0).float()
            pruning_masks[name] = {
                'weight': active.view(-1, 1, 1, 1).expand_as(w).clone(),
                'bias': active.clone() if module.bias is not None else None
            }

    eff_params = sum((p.abs() > 0).sum().item() for p in pruned_gen.parameters())
    all_params = sum(p.numel() for p in pruned_gen.parameters())
    print(f"Effective params: {eff_params:,} / {all_params:,} ({eff_params/all_params*100:.1f}%)")
    print(f"Theoretical size if channels were physically removed: {eff_params * 4 / 1e6:.1f} MB")

    # 2. evaluate before fine tuning
    pruned_gen.cuda().eval()
    if os.path.exists(pre_ft_path):
        with open(pre_ft_path) as f:
            pre_ft_results = json.load(f)
        print("Loaded cached pre-fine-tuning results.")
    else:
        print(f"\nEvaluating {ratio*100:.0f}% pruned (before fine-tuning)...")
        pre_ft_results, _, _ = evaluate(pruned_gen, test_loader, desc=f'{tag} pre-FT')
        with open(pre_ft_path, 'w') as f:
            json.dump(pre_ft_results, f, indent=2)

    # 3. fine tune
    discriminator = PatchDiscriminator().cuda()
    vgg_loss = VGGPerceptualLoss().cuda()
    g_opt = torch.optim.Adam(pruned_gen.parameters(), lr=LEARNING_RATE_G, betas=(0.5, 0.999))
    d_opt = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE_D, betas=(0.5, 0.999))

    history = {'epoch': [], 'g_loss': [], 'd_loss': [],
               'val_epoch': [], 'val_psnr': [], 'val_ssim': []}

    # resume from the latest checkpoint if any
    start_epoch = 0
    if os.path.exists(latest_ckpt):
        ckpt = torch.load(latest_ckpt, map_location='cuda', weights_only=False)
        pruned_gen.load_state_dict(ckpt['generator'])
        discriminator.load_state_dict(ckpt['discriminator'])
        g_opt.load_state_dict(ckpt['g_opt'])
        d_opt.load_state_dict(ckpt['d_opt'])
        start_epoch = ckpt['epoch'] + 1
        if os.path.exists(history_path):
            with open(history_path) as f:
                history = json.load(f)
        print(f"Resumed from epoch {ckpt['epoch']+1}, continuing at epoch {start_epoch+1}")

    # keep track of the best validation score
    best_psnr = max(history['val_psnr']) if history['val_psnr'] else -float('inf')

    print(f"\nFine-tuning {tag}: epochs {start_epoch+1}-{NUM_EPOCHS}, {len(train_loader)} batches/epoch")
    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_start = time.time()
        pruned_gen.train()
        discriminator.train()
        g_losses, d_losses = [], []

        for i, (blur, sharp) in enumerate(train_loader):
            blur, sharp = blur.cuda(), sharp.cuda()
            fake = pruned_gen(blur)

            d_opt.zero_grad()
            d_loss = d_loss_fn(discriminator(sharp), discriminator(fake.detach()))
            d_loss.backward()
            d_opt.step()

            g_opt.zero_grad()
            fake_pred = discriminator(fake)
            real_pred = discriminator(sharp).detach()
            g_adv = g_loss_fn(real_pred, fake_pred)
            g_perc = vgg_loss(fake, sharp)
            g_pix = F.mse_loss(fake, sharp)
            g_loss = PIXEL_WEIGHT * g_pix + PERCEPTUAL_WEIGHT * g_perc + ADVERSARIAL_WEIGHT * g_adv
            g_loss.backward()
            g_opt.step()
            apply_masks(pruned_gen, pruning_masks)

            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

            if (i + 1) % 200 == 0:
                print(f"  Epoch {epoch+1}/{NUM_EPOCHS} [{i+1}/{len(train_loader)}] "
                      f"G: {np.mean(g_losses[-200:]):.4f}  D: {np.mean(d_losses[-200:]):.4f}")

        avg_g, avg_d = float(np.mean(g_losses)), float(np.mean(d_losses))
        history['epoch'].append(epoch + 1)
        history['g_loss'].append(avg_g)
        history['d_loss'].append(avg_d)

        # validate every few epochs and save the best model separately
        val_msg = ''
        if (epoch + 1) % VAL_EVERY == 0 or epoch == NUM_EPOCHS - 1:
            val_psnr, val_ssim = validate(pruned_gen, val_loader)
            history['val_epoch'].append(epoch + 1)
            history['val_psnr'].append(val_psnr)
            history['val_ssim'].append(val_ssim)
            val_msg = f"  val PSNR: {val_psnr:.2f} dB  val SSIM: {val_ssim:.4f}"
            if val_psnr > best_psnr:
                best_psnr = val_psnr
                tmp_best = best_ckpt + '.tmp'
                torch.save(pruned_gen.state_dict(), tmp_best)
                os.replace(tmp_best, best_ckpt)
                val_msg += '  (new best, saved)'

        mins = (time.time() - epoch_start) / 60
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} done in {mins:.1f} min | G: {avg_g:.4f}  D: {avg_d:.4f}{val_msg}")

        # save the latest checkpoint after every epoch
        tmp_path = latest_ckpt + '.tmp'
        torch.save({
            'epoch': epoch,
            'generator': pruned_gen.state_dict(),
            'discriminator': discriminator.state_dict(),
            'g_opt': g_opt.state_dict(),
            'd_opt': d_opt.state_dict(),
        }, tmp_path)
        os.replace(tmp_path, latest_ckpt)
        with open(history_path, 'w') as f:
            json.dump(history, f, indent=2)

    # 4. evaluate the best checkpoint after fine tuning
    best_epoch, best_val_psnr = None, None
    if history['val_psnr']:
        k = int(np.argmax(history['val_psnr']))
        best_epoch, best_val_psnr = history['val_epoch'][k], history['val_psnr'][k]
    if os.path.exists(best_ckpt):
        pruned_gen.load_state_dict(torch.load(best_ckpt, map_location='cuda', weights_only=False))
        print(f"\nLoaded best checkpoint: epoch {best_epoch} (val PSNR {best_val_psnr:.2f} dB)")
    pruned_gen.eval()
    print(f"Evaluating {ratio*100:.0f}% pruned (after fine-tuning, best epoch)...")
    post_ft_results, _, _ = evaluate(pruned_gen, test_loader, desc=f'{tag} post-FT')

    # 5. save results
    all_results[tag] = {
        'ratio': ratio,
        'before_ft': pre_ft_results,
        'after_ft': post_ft_results,
        'best_epoch': best_epoch,
        'best_val_psnr': best_val_psnr,
        'eff_params': eff_params,
        'total_params': all_params,
        'theoretical_size_mb': eff_params * 4 / 1e6,
    }
    with open(results_path, 'w') as f:
        json.dump(all_results[tag], f, indent=2)
    torch.save(pruned_gen.state_dict(), final_model_path)

    # cleanup
    del pruned_gen, discriminator, vgg_loss, g_opt, d_opt
    torch.cuda.empty_cache()
    print(f"\n{tag} complete.")

print(f"\n{'='*60}")
print("All pruning ratios complete!")
print(f"{'='*60}")

## 8. Results and visualizations

PSNR and SSIM in the table below come from the clean single-session re-measurement in Section 9 (scores drift between Colab sessions, so per-session numbers are not comparable). Efficiency metrics and training info (latency, memory, size, best epochs, effective parameters) come from the run itself. The quality-vs-ratio figure is drawn in Section 9.


In [ ]:
import matplotlib.pyplot as plt

# sort by ratio
tags = sorted(all_results, key=lambda t: all_results[t]['ratio'])

# PSNR/SSIM come from the clean single-session re-measurement (Section 9) when it exists,
# because the per-session values are not comparable across Colab sessions
clean_path = os.path.join(CHECKPOINT_BASE, 'results_clean_full_train.json')
clean = None
if os.path.exists(clean_path):
    with open(clean_path) as f:
        clean = json.load(f)

def quality(tag, metric):
    if clean:
        key = 'baseline' if tag == 'baseline' else tag
        return clean[key][metric]
    return baseline_results[metric] if tag == 'baseline' else all_results[tag]['after_ft'][metric]

# comparison table
print("\n" + "=" * 90)
print("RESULTS: Baseline vs All Pruning Ratios (After Fine-Tuning)")
if clean:
    print("PSNR/SSIM from the clean single-session re-measurement (Section 9)")
else:
    print("WARNING: clean re-measurement not found, PSNR/SSIM are per-session values")
print("=" * 90)

header = f"{'Metric':<25} {'Baseline':>12}"
for tag in tags:
    header += f" {tag:>12}"
print(header)
print("-" * 90)

for metric, label, fmt in [('psnr', 'PSNR (dB)', '.2f'), ('ssim', 'SSIM', '.4f')]:
    row = f"{label:<25} {quality('baseline', metric):>12{fmt}}"
    for tag in tags:
        row += f" {quality(tag, metric):>12{fmt}}"
    print(row)

for metric, label, fmt in [('latency_ms', 'Latency (ms)', '.1f'),
                           ('peak_memory_gb', 'Peak GPU Mem (GB)', '.2f'),
                           ('model_size_mb', 'Model Size (MB)', '.1f')]:
    row = f"{label:<25} {baseline_results[metric]:>12{fmt}}"
    for tag in tags:
        row += f" {all_results[tag]['after_ft'][metric]:>12{fmt}}"
    print(row)

print("=" * 90)

# best epoch per ratio
print("\nBest validation epoch per ratio:")
for tag in tags:
    res = all_results[tag]
    if res.get('best_epoch') is not None:
        print(f"  {tag}: epoch {res['best_epoch']} (val PSNR {res['best_val_psnr']:.2f} dB)")

# effective parameter counts
print("\nEffective parameters:")
for tag in tags:
    res = all_results[tag]
    pct = res['eff_params'] / res['total_params'] * 100
    print(f"  {tag}: {res['eff_params']:,} / {res['total_params']:,} ({pct:.1f}%) "
          f"-> theoretical size {res['theoretical_size_mb']:.1f} MB")

# PSNR changes
print("\nPSNR change from baseline:")
for tag in tags:
    delta = quality(tag, 'psnr') - quality('baseline', 'psnr')
    print(f"  {tag}: {delta:+.2f} dB")

# the PSNR/SSIM vs ratio figure is drawn in Section 9 from the clean results
# (fig_quality_vs_pruning_ratio.png); the old per-session plot was misleading and is gone


In [ ]:
# convergence curves for each ratio
fig, ax = plt.subplots(figsize=(9, 5.5))

for tag in sorted(all_results, key=lambda t: all_results[t]['ratio']):
    hist_path = os.path.join(CHECKPOINT_BASE, f'ratio_{tag}', 'history.json')
    if not os.path.exists(hist_path):
        print(f"No history for {tag}, skipping")
        continue
    with open(hist_path) as f:
        hist = json.load(f)
    ax.plot(hist['val_epoch'], hist['val_psnr'], 'o-', linewidth=2,
            label=f"Pruned {int(all_results[tag]['ratio']*100)}%")

ax.axhline(baseline_results['psnr'], color='gray', linestyle='--',
           label='Baseline (unpruned, full test set)')
ax.set_xlabel('Fine-tuning epoch', fontsize=12)
ax.set_ylabel(f'Validation PSNR (dB, {VAL_IMAGES}-image subset)', fontsize=12)
ax.set_title('Fine-tuning convergence by pruning ratio', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'fine_tuning_convergence.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# side by side comparison images
tags = sorted(all_results, key=lambda t: all_results[t]['ratio'])
pruned_models = {}
for tag in tags:
    ckpt_dir = os.path.join(CHECKPOINT_BASE, f'ratio_{tag}')
    model_path = os.path.join(ckpt_dir, f'pruned_{tag}_finetuned.pth')
    model = copy.deepcopy(generator)
    model.load_state_dict(torch.load(model_path, map_location='cuda', weights_only=False))
    model.eval()
    pruned_models[tag] = model

n_cols = 3 + len(tags)  # Blurry, Sharp, Baseline, + one per ratio
indices = np.linspace(0, len(test_dataset) - 1, 6, dtype=int)

fig, axes = plt.subplots(6, n_cols, figsize=(5 * n_cols, 30))
col_titles = ['Blurry Input', 'Sharp (GT)', 'Baseline'] + \
             [f"Pruned {int(all_results[t]['ratio']*100)}%" for t in tags]

to_img = lambda t: ((t.squeeze().cpu().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)

for row, idx in enumerate(indices):
    blur, sharp = test_dataset[idx]
    blur_gpu = blur.unsqueeze(0).cuda()
    _, _, h, w = blur_gpu.shape
    pad_h = (32 - h % 32) % 32
    pad_w = (32 - w % 32) % 32
    blur_pad = F.pad(blur_gpu, (0, pad_w, 0, pad_h), mode='reflect')

    with torch.no_grad():
        base_out = generator(blur_pad)[:, :, :h, :w]

    imgs = [to_img(blur), to_img(sharp), to_img(base_out)]
    gt = imgs[1]
    titles = [col_titles[0], col_titles[1],
              f'{col_titles[2]}\nPSNR: {peak_signal_noise_ratio(gt, imgs[2]):.2f} dB']

    for j, tag in enumerate(tags):
        with torch.no_grad():
            out = pruned_models[tag](blur_pad)[:, :, :h, :w]
        img = to_img(out)
        imgs.append(img)
        titles.append(f'{col_titles[3+j]}\nPSNR: {peak_signal_noise_ratio(gt, img):.2f} dB')

    for col in range(n_cols):
        axes[row, col].imshow(imgs[col])
        axes[row, col].set_title(titles[col], fontsize=10, fontweight='bold' if row == 0 else 'normal')
        axes[row, col].axis('off')

plt.tight_layout()
# jpg because the png was too big
fig_path = os.path.join(FIGURES_DIR, 'side_by_side_comparison.jpg')
plt.savefig(fig_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

# free the models
del pruned_models
torch.cuda.empty_cache()

In [ ]:
# save everything
save_data = {
    'baseline': baseline_results,
    'config': {
        'pruning_ratios': sorted(PRUNING_RATIOS),
        'num_epochs': NUM_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr_g': LEARNING_RATE_G,
        'lr_d': LEARNING_RATE_D,
        'crop_size': CROP_SIZE,
        'pixel_weight': PIXEL_WEIGHT,
        'perceptual_weight': PERCEPTUAL_WEIGHT,
        'adversarial_weight': ADVERSARIAL_WEIGHT,
        'dataset': 'GoPro official split (2,103 train / 1,111 test)',
        'val_every': VAL_EVERY,
        'val_images': VAL_IMAGES,
    }
}
for tag in sorted(all_results, key=lambda t: all_results[t]['ratio']):
    res = all_results[tag]
    save_data[f'{tag}_before_finetuning'] = res['before_ft']
    save_data[f'{tag}_after_finetuning'] = res['after_ft']
    save_data[f'{tag}_best_epoch'] = res.get('best_epoch')
    save_data[f'{tag}_best_val_psnr'] = res.get('best_val_psnr')
    save_data[f'{tag}_eff_params'] = res['eff_params']
    save_data[f'{tag}_total_params'] = res['total_params']
    save_data[f'{tag}_theoretical_size_mb'] = res['theoretical_size_mb']

results_path = os.path.join(CHECKPOINT_BASE, 'experiment_results.json')
with open(results_path, 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"Results: {results_path}")
print(f"Figures: {FIGURES_DIR}/")
print(f"Models saved in: {CHECKPOINT_BASE}/ratio_*/")
print("\nDone!")

## Done

Everything is saved to Google Drive: the metrics for every ratio, the model weights, the training history and the figures. The old v1 run is kept in its own folder for reference.


## 9. Diagnostics

Two extra checks I had to run before I could trust the results.

A. My baseline scored well below the published number. This check found the reason: the official code runs the model in training mode, and in that mode the score matches the paper.

B. Scores also changed between Colab sessions, so this cell measures all seven models again in one single session. These are the numbers used in the thesis.


In [ ]:
# Diagnostic A: check why the baseline scores lower than the published number
import numpy as np, cv2, torch, torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio

idx10 = np.linspace(0, len(test_blur) - 1, 10, dtype=int)
diag_pairs = [(test_blur[i], test_sharp[i]) for i in idx10]

def diag_score(model, train_mode):
    model.train(train_mode) if train_mode else model.eval()
    psnrs = []
    with torch.no_grad():
        for bp, sp in diag_pairs:
            blur = cv2.cvtColor(cv2.imread(bp), cv2.COLOR_BGR2RGB).astype(np.float32) / 127.5 - 1.0
            x = torch.from_numpy(blur.transpose(2, 0, 1))[None].cuda()
            _, _, h, w = x.shape
            ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
            xp = F.pad(x, (0, pw, 0, ph), mode='reflect') if (ph or pw) else x
            out = model(xp)[:, :, :h, :w]
            out = ((out.squeeze().cpu().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            gt = cv2.cvtColor(cv2.imread(sp), cv2.COLOR_BGR2RGB)
            psnrs.append(peak_signal_noise_ratio(gt, out))
    model.eval()
    return float(np.mean(psnrs))

print(f"our protocol, eval() mode : {diag_score(generator, False):6.2f} dB   (protocol behind all cached results)")
print(f"our protocol, train() mode: {diag_score(generator, True):6.2f} dB   (predict.py's inference mode)")

# also run the official predict pipeline on the same images
try:
    import sys, os
    if '/content/DeblurGANv2' not in sys.path:
        sys.path.insert(0, '/content/DeblurGANv2')
    _cwd = os.getcwd(); os.chdir('/content/DeblurGANv2')
    from predict import Predictor
    official = Predictor(weights_path=WEIGHTS_PATH)
    psnrs = []
    for bp, sp in diag_pairs:
        out = official(cv2.imread(bp), None)   # predict.py's main() feeds cv2.imread output
        gt_rgb = cv2.cvtColor(cv2.imread(sp), cv2.COLOR_BGR2RGB)
        gt_bgr = cv2.imread(sp)
        # try both channel orders and keep the better one
        psnrs.append(max(peak_signal_noise_ratio(gt_rgb, out), peak_signal_noise_ratio(gt_bgr, out)))
    os.chdir(_cwd)
    print(f"official predict.py       : {np.mean(psnrs):6.2f} dB")
except Exception as e:
    os.chdir(_cwd)
    print(f"official Predictor failed ({type(e).__name__}: {e}) - rely on the train/eval comparison above")

print("""
How to read this:
  - train() ~= eval(), both ~24-25 dB, official also ~24-25 dB:
      The released checkpoint simply scores lower under a faithful protocol.
      Report the reproduced baseline honestly; cite the repo's known issues.
  - train() >> eval() (e.g. ~29 vs ~25):
      Root cause found: BatchNorm running stats in the backbone. The official
      protocol is train-mode inference. Rerun the four final evals in train mode
      (Diagnostic B with EVAL_TRAIN_MODE = True) and use those numbers.
""")

In [ ]:
# Diagnostic B: re-measure all seven models in one session since scores vary between Colab sessions

EVAL_TRAIN_MODE = True

import copy, glob, json, time, tempfile, os
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm import tqdm

mode_tag = 'train' if EVAL_TRAIN_MODE else 'eval'
out_path = os.path.join(CHECKPOINT_BASE, f'results_clean_full_{mode_tag}.json')

def evaluate_clean(model, loader, train_mode=False, desc='Evaluating', warmup=5):
    model.train(train_mode) if train_mode else model.eval()
    psnr_list, ssim_list, lat = [], [], []
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        for blur, sharp in tqdm(loader, desc=desc):
            blur = blur.cuda()
            _, _, h, w = blur.shape
            ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
            xp = F.pad(blur, (0, pw, 0, ph), mode='reflect') if (ph or pw) else blur
            torch.cuda.synchronize(); t0 = time.time()
            out = model(xp)
            torch.cuda.synchronize(); lat.append((time.time() - t0) * 1000)
            out = out[:, :, :h, :w]
            o = ((out.squeeze().cpu().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            g = ((sharp.squeeze().numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            psnr_list.append(peak_signal_noise_ratio(g, o))
            ssim_list.append(structural_similarity(g, o, channel_axis=2))
    model.eval()
    with tempfile.NamedTemporaryFile(suffix='.pth') as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / 1e6
    timed = lat[warmup:] if len(lat) > warmup else lat
    return {
        'psnr': float(np.mean(psnr_list)), 'ssim': float(np.mean(ssim_list)),
        'latency_ms': float(np.mean(timed)),
        'peak_memory_gb': float(torch.cuda.max_memory_allocated() / 1e9),
        'model_size_mb': float(size_mb),
        'per_image_psnr': [float(x) for x in psnr_list],
        'per_image_ssim': [float(x) for x in ssim_list],
    }

def make_pruned(ratio):
    # same pruning as the main loop
    m = copy.deepcopy(generator)
    for module in m.modules():
        if isinstance(module, nn.Conv2d) and module.out_channels > 3:
            w = module.weight.data
            n_zero = int(w.shape[0] * ratio)
            if n_zero == 0:
                continue
            _, idx = torch.topk(w.abs().sum(dim=[1, 2, 3]), n_zero, largest=False)
            w[idx] = 0
            if module.bias is not None:
                module.bias.data[idx] = 0
    return m.cuda().eval()

# this measurement takes hours, so load the saved one if it already exists
if os.path.exists(out_path):
    with open(out_path) as f:
        clean = json.load(f)
    print(f"Loaded existing clean results: {out_path}")
    print(f"(measured on {clean.get('gpu', 'unknown GPU')}, {clean.get('eval_mode', '?')} mode; "
          "delete the file on Drive to re-measure)")
else:
    clean = {'eval_mode': mode_tag, 'gpu': torch.cuda.get_device_name(0)}

    clean['baseline'] = evaluate_clean(generator, test_loader, EVAL_TRAIN_MODE, desc='Baseline (clean)')

    for ratio in sorted(PRUNING_RATIOS):
        tag = f"{int(ratio*100)}pct"
        m = make_pruned(ratio)
        clean[f'{tag}_preft'] = evaluate_clean(m, test_loader, EVAL_TRAIN_MODE, desc=f'{tag} pre-FT (clean)')
        del m; torch.cuda.empty_cache()
        cands = glob.glob(os.path.join(CHECKPOINT_BASE, '**', f'*{tag}*finetuned*.pth'), recursive=True)
        assert cands, f"No fine-tuned weights found for {tag} under {CHECKPOINT_BASE}"
        m = copy.deepcopy(generator)
        m.load_state_dict(torch.load(cands[0], map_location='cuda'))
        clean[tag] = evaluate_clean(m, test_loader, EVAL_TRAIN_MODE, desc=f'{tag} post-FT (clean)')
        del m; torch.cuda.empty_cache()

    with open(out_path, 'w') as f:
        json.dump(clean, f, indent=2)
    print(f"\nSaved: {out_path}")

cols = ['baseline'] + [c for r in sorted(PRUNING_RATIOS)
                       for c in (f"{int(r*100)}pct_preft", f"{int(r*100)}pct")]
print(f"\n{'Metric':<16}" + ''.join(f"{c:>14}" for c in cols))
print('-' * (16 + 14 * len(cols)))
for key, fmt in [('psnr', '.2f'), ('ssim', '.4f'), ('latency_ms', '.1f'),
                 ('peak_memory_gb', '.2f'), ('model_size_mb', '.1f')]:
    print(f"{key:<16}" + ''.join(f"{clean[c][key]:>14{fmt}}" for c in cols))

In [ ]:
# draw the report figure from the clean session results

import json, os
import matplotlib.pyplot as plt

with open(os.path.join(CHECKPOINT_BASE, 'results_clean_full_train.json')) as f:
    clean = json.load(f)

pcts = [0] + sorted(int(r * 100) for r in PRUNING_RATIOS)
psnr_after = [clean['baseline']['psnr']] + [clean[f'{p}pct']['psnr'] for p in pcts[1:]]
psnr_before = [clean['baseline']['psnr']] + [clean[f'{p}pct_preft']['psnr'] for p in pcts[1:]]
ssim_after = [clean['baseline']['ssim']] + [clean[f'{p}pct']['ssim'] for p in pcts[1:]]
ssim_before = [clean['baseline']['ssim']] + [clean[f'{p}pct_preft']['ssim'] for p in pcts[1:]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(pcts, psnr_after, 'bo-', linewidth=2.5, markersize=10, label='After fine-tuning')
ax1.plot(pcts, psnr_before, 'o--', color='gray', linewidth=2, markersize=10, label='Before fine-tuning')
ax1.set_xlabel('Pruning Ratio (%)', fontsize=12)
ax1.set_ylabel('PSNR (dB)', fontsize=12)
ax1.set_title('PSNR vs Pruning Ratio', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(pcts, ssim_after, 'ro-', linewidth=2.5, markersize=10, label='After fine-tuning')
ax2.plot(pcts, ssim_before, 'o--', color='gray', linewidth=2, markersize=10, label='Before fine-tuning')
ax2.set_xlabel('Pruning Ratio (%)', fontsize=12)
ax2.set_ylabel('SSIM', fontsize=12)
ax2.set_title('SSIM vs Pruning Ratio', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_quality_vs_pruning_ratio.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_quality_vs_pruning_ratio.png")
